<a href="https://colab.research.google.com/github/kavyaaaa16/Heart-Disease-Predictor/blob/main/BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install transformers datasets --quiet

import pandas as pd
import tensorflow as tf
from transformers import BertTokenizerFast, TFBertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical


In [26]:
twitter=pd.read_csv('/content/twitter_training.csv')

In [27]:
twitter.columns=['id', 'category','label', 'review']
twitter

,id,category,label,review
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [28]:
twitter.isnull().sum()

,0
id,0
category,0
label,0
review,686


In [29]:
twitter.dropna(subset=['review'],inplace=True)
twitter.isnull().sum()

,0
id,0
category,0
label,0
review,0


In [30]:
twitter.drop(['id', 'category'], axis='columns')

,label,review
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...
...,...,...
74676,Positive,Just realized that the Windows partition of my...
74677,Positive,Just realized that my Mac window partition is ...
74678,Positive,Just realized the windows partition of my Mac ...
74679,Positive,Just realized between the windows partition of...


In [31]:
pip install emoji

In [32]:
import re
import emoji

def cleaning(text):
  text=text.lower()
  text=emoji.replace_emoji(text, replace='')
  text=re.sub(r"http\S+", "", text)
  text= re.sub(r"[^a-zA-Z0-9\s]", "", text)
  text=re.sub(r"(.)\1{2,}", r"\1", text)
  text=re.sub(r"\s+", " ", text).strip()
  return text

twitter['review']=twitter['review'].astype(str).apply(cleaning)

In [33]:
valid_labels=['Positive', 'Negative', 'Neutral']
twitter=twitter[twitter['label'].isin(valid_labels)]
twitter=twitter.reset_index(drop=True)

In [34]:
twitter.shape

(61120, 4)

In [35]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

topword_list=stopwords.words('english') #will give a list of stopwords
stop=set(stopwords.words('english'))  #we convert it to set for faster checking
print(stop)

{'once', 'and', 'myself', 'at', "shan't", 'here', 'with', 'in', 'had', 'was', 'd', 'did', 'of', 'why', 'which', "aren't", 'be', 'me', 's', 'no', 'or', 'most', 'himself', 'm', 'mustn', 'only', 'him', 'where', "didn't", 'don', 'that', 'than', 'above', 'if', 'he', 'whom', 'theirs', 'mightn', 'on', 'both', "i've", 'these', "i'll", 'his', 'ours', 'our', 'because', 'this', "couldn't", 'all', 'am', 'hasn', 'any', "i'd", 'what', 'from', 'other', "we're", 'them', "needn't", 'y', 'she', "hadn't", 'now', "it'll", "she's", 'isn', 've', 'we', 'aren', 'down', 'hadn', "they'd", 'they', 'further', 'didn', 'there', 'couldn', 'over', 'before', 'but', 'it', 'until', "shouldn't", 'doing', 'again', 'by', "isn't", 'can', "wasn't", 'yourselves', "he'd", 'weren', 'have', 'doesn', 'as', 'those', 'ain', 'yourself', "we've", 'just', 'been', 'up', "we'd", 'itself', 'an', "wouldn't", "you'll", "hasn't", "weren't", "should've", 're', 'so', 'very', 'more', "they'll", 't', "she'll", 'into', 'haven', 'ma', "they've", 

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
def remove_stopwords(text):
  words=text.split()
  filtered=[w for w in words if w not in stop]
  return ' '.join(filtered)

twitter['review']=twitter['review'].apply(remove_stopwords)

In [37]:
test_df=pd.read_csv('/content/books_test.csv')

In [38]:
test_df=test_df.drop(['title','author', 'rating','rating_count', 'genre'], axis='columns')


In [39]:
test_df['review']=test_df['review'].apply(remove_stopwords)
test_df

,review,label
0,ive reading lot romance lately thanks arcs kno...,positive
1,book shows us become supercommunicators superc...,positive
2,tldr empire ai definitive chronicle ai revolut...,negative
3,note 4172023 someone reported unmarked spoiler...,negative
4,lovely lovely book poems nearly every poem end...,positive
...,...,...
194,rarely write reviews book feels like personal ...,negative
195,keep switching rating book 5 4 5 changing opin...,positive
196,absolutely loved rereading pet sematary experi...,positive
197,rating stars universe needed dead agreedyou ne...,positive


In [40]:
# Fix label typos and standardize to lowercase
def clean_label(label):
    label = label.lower()
    if label == 'neagtive':
        return 'negative'
    return label

twitter['label'] = twitter['label'].apply(clean_label)
test_df['label'] = test_df['label'].apply(clean_label)


In [65]:
def head_tail_tokenize(texts, tokenizer, head_len=96, tail_len=32, max_length=128):
    all_input_ids = []
    all_attention_mask = []

    for text in texts:
        tokens = tokenizer.tokenize(text)

        if len(tokens) > max_length:
            head_tokens = tokens[:head_len]
            tail_tokens = tokens[-tail_len:]
            combined_tokens = head_tokens + tail_tokens
        else:
            combined_tokens = tokens

        # Convert tokens to ids
        input_ids = tokenizer.convert_tokens_to_ids(combined_tokens)

        # Add special tokens ([CLS] and [SEP]) if tokenizer uses them
        # For BERT-style models, usually:
        input_ids = [tokenizer.cls_token_id] + input_ids + [tokenizer.sep_token_id]

        # Pad or truncate to max_length
        if len(input_ids) > max_length:
            input_ids = input_ids[:max_length]
        else:
            input_ids += [tokenizer.pad_token_id] * (max_length - len(input_ids))

        # Attention mask: 1 for real tokens, 0 for padding
        attention_mask = [1 if id != tokenizer.pad_token_id else 0 for id in input_ids]

        all_input_ids.append(input_ids)
        all_attention_mask.append(attention_mask)

    return {'input_ids': all_input_ids, 'attention_mask': all_attention_mask}


In [61]:
train_texts = twitter['review'].astype(str).tolist()
train_labels = twitter['label'].tolist()

test_texts = test_df['review'].astype(str).tolist()
test_labels = test_df['label'].tolist()


In [66]:

# Convert both label lists to string
train_labels = [str(label) for label in train_labels]
test_labels = [str(label) for label in test_labels]

# Combine labels before fitting
all_labels = train_labels + test_labels

label_encoder = LabelEncoder()
label_encoder.fit(all_labels)

# Then transform
y_train = to_categorical(label_encoder.transform(train_labels))
y_test = to_categorical(label_encoder.transform(test_labels))



In [73]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = head_tail_tokenize(train_texts, tokenizer, head_len=96, tail_len=32, max_length=64)
test_encodings = head_tail_tokenize(test_texts, tokenizer, head_len=96, tail_len=32, max_length=64)



In [74]:
import tensorflow as tf
batch_size = 16
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    y_train
)).shuffle(len(train_texts)).batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    y_test
)).batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)





In [75]:
from transformers import TFDistilBertForSequenceClassification
model = TFDistilBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


You are using a model of type bert to instantiate a model of type distilbert. This is not supported for all configurations of models and can yield errors.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['bert.encoder.layer.11.attention.self.query.bias', 'bert.encoder.layer.1.attention.output.dense.bias', 'bert.encoder.layer.8.attention.output.dense.bias', 'bert.encoder.layer.7.attention.output.dense.weight', 'bert.encoder.layer.10.attention.self.query.bias', 'bert.encoder.layer.9.output.LayerNorm.weight', 'bert.encoder.layer.11.attention.self.key.bias', 'bert.encoder.layer.6.attention.self.query.weight', 'bert.encoder.layer.3.output.dense.bias', 'bert.encoder.layer.1.attention.self.value.weight', 'bert.encoder.layer.10.output.dense.weight', 'bert.encoder.layer.8.attention.self.value.bias', 'bert.embeddings.word_embeddings.weight', 'bert.encoder.layer.11.attention.self.value.bias', 'bert.encoder.layer.0.attention

In [76]:
import tensorflow as tf

loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
metrics = ['accuracy']


In [77]:
from transformers import TFBertForSequenceClassification
from transformers import AdamWeightDecay
from tensorflow.keras.losses import CategoricalCrossentropy

# Load model
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Create optimizer with Hugging Face-compatible AdamW
optimizer = AdamWeightDecay(learning_rate=3e-5, weight_decay_rate=0.01)

# Set loss and metrics
loss = CategoricalCrossentropy(from_logits=True)
metrics = ['accuracy']

# Compile model (this will now work)
model.compile(optimizer=optimizer, loss=loss, metrics=metrics)


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [78]:
epochs = 3

model.fit(train_dataset, validation_data=test_dataset, epochs=1)


 426/3820 [==>...........................] - ETA: 11:45:54 - loss: 0.8672 - accuracy: 0.6036

KeyboardInterrupt: 

In [79]:
model.save_pretrained('/content/sentiment_bert_model_headtail')
tokenizer.save_pretrained('/content/sentiment_bert_model_headtail')


('/content/sentiment_bert_model_headtail/tokenizer_config.json',
 '/content/sentiment_bert_model_headtail/special_tokens_map.json',
 '/content/sentiment_bert_model_headtail/vocab.txt',
 '/content/sentiment_bert_model_headtail/added_tokens.json')

In [49]:
model.save_pretrained("model_checkpoint_v1")
tokenizer.save_pretrained("model_checkpoint_v1")


('model_checkpoint_v1/tokenizer_config.json',
 'model_checkpoint_v1/special_tokens_map.json',
 'model_checkpoint_v1/vocab.txt',
 'model_checkpoint_v1/added_tokens.json')

In [80]:
model.evaluate(test_dataset)


13/13 [==============================] - 53s 4s/step - loss: 0.8811 - accuracy: 0.6683


[0.8811438679695129, 0.6683416962623596]

In [ ]:
weak_df=pd.read_csv('/content/train_labeled.csv')
weak_df=weak_df.drop(['title','author', 'rating','rating_count', 'genre'], axis='columns')
weak_df['label'] = weak_df['label'].str.strip().str.capitalize()
# --- Step 1: Prepare your weak dataset ---
# Convert reviews to strings (if needed)
weak_df['review'] = weak_df['review'].astype(str)
weak_df = weak_df[weak_df['label'].isin(['Positive', 'Negative', 'Neutral'])]
weak_df = weak_df.dropna(subset=['review'])
weak_df = weak_df[weak_df['review'].str.strip() != '']
weak_df = weak_df.reset_index(drop=True)

# Apply preprocessing
weak_df['review'] = weak_df['review'].apply(remove_stopwords)
weak_df['label'] = weak_df['label'].apply(clean_label)




# Tokenize
weak_encodings = tokenizer(
    weak_df['review'].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors='tf'
)

# --- Step 2: Encode weak labels (if available) ---
if 'label' in weak_df.columns:
    weak_labels = [str(lab) for lab in weak_df['label']]
    encoded_labels = label_encoder.transform(weak_labels)
    y_weak = to_categorical(encoded_labels)
    weak_dataset = tf.data.Dataset.from_tensor_slices((dict(weak_encodings), y_weak))
else:
    # For prediction-only mode
    weak_dataset = tf.data.Dataset.from_tensor_slices(dict(weak_encodings))

# Batch it
weak_dataset = weak_dataset.batch(16).cache().prefetch(tf.data.AUTOTUNE)

# --- Step 3: Evaluate or Predict ---
if 'label' in weak_df.columns:
    # Evaluate if labels are available
    result = model.evaluate(weak_dataset)
    print(f"Weak Label Evaluation - Loss: {result[0]}, Accuracy: {result[1]}")
else:
    # Just predict if labels not available
    predictions = model.predict(weak_dataset).logits
    predicted_labels = np.argmax(predictions, axis=1)
    decoded_labels = label_encoder.inverse_transform(predicted_labels)
    weak_df['predicted_label'] = decoded_labels
    print(weak_df[['review', 'predicted_label']].head())


 1/71 [..............................] - ETA: 13:49 - loss: 0.8006 - accuracy: 0.5625